# 07｜空間分析與可及性：距離要在公尺下算

**115-1 地理資訊系統運用程式｜W7（10/29）**

W6 我們用座標判斷站點「在哪一區」。這週要回答「**離多遠**」與「**範圍內有幾個**」：火車站、捷運站附近，有沒有 YouBike 可以轉乘？

> **buffer 與距離不會因為 CRS 錯了就報錯。** 在經緯度下 `buffer(500)`，畫出來的是半徑 500「度」的圓，比整個地球還大。可及性分析的每一個數字，都要先確認單位是公尺。

| Part | 內容 |
|---|---|
| A | 取得軌道車站（臺鐵、高鐵、臺中捷運）與臺中市 YouBike 站點 |
| B | 合併三種車站，用 `sjoin` 篩出臺中市（臺鐵沒有縣市欄位） |
| C | 環域（buffer）與單位：經緯度和 EPSG:3826 差在哪 |
| D | 環域內有幾個 YouBike 站：300 公尺與 500 公尺 |
| E | 最近鄰連接 `sjoin_nearest`：每個車站到最近站點的距離 |
| F | 臺中的答案是 0：換指標、換縣市 |
| G | `dissolve` 與 `overlay`：環域重疊與各區覆蓋比例 |
| H | 什麼算「一個車站」，以及直線距離的限制 |

**課前準備：**

1. W6 的 `Taiwan_boundaries` 與 `tdx_station_20261012` 兩個資料夾，應該已經在雲端硬碟的 `115-1_GISprog/data/raw/`。
2. 從新雲端學院下載 `W7_課堂資料.zip`，解壓縮後，把 `tdx_rail_20261023` 資料夾上傳到同一個位置。
3. Colab Secrets 的 `TDX_CLIENT_ID`、`TDX_CLIENT_SECRET` 沿用 W6。還沒有金鑰的同學，Part A 會自動改讀老師的備援快照。

標示 **【練習】** 的儲存格請自己動手完成。

## 0. 路徑與中文字型

沿用 W6 的兩個儲存格，多了本週軌道車站的備援快照路徑。

In [ ]:
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = Path("/content/drive/MyDrive/115-1_GISprog")
else:
    # 備援：本機執行時，notebook 位於 115-1_GISprog/notebooks/，上一層就是專案資料夾
    PROJECT_DIR = Path.cwd().parent

RAW_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
FONT_DIR = PROJECT_DIR / "fonts"

TOWN_SHP = RAW_DIR / "Taiwan_boundaries" / "Town" / "Taiwan_town.shp"  # 鄉鎮市區界
BIKE_BACKUP = RAW_DIR / "tdx_station_20261012"                         # W6 的站點備援快照
RAIL_BACKUP = RAW_DIR / "tdx_rail_20261023"                            # 本週的軌道車站備援快照
TDX_DIR = RAW_DIR / "tdx"                                              # 你自己從 TDX 抓的資料

for p in [TOWN_SHP, BIKE_BACKUP, RAIL_BACKUP]:
    if not p.exists():
        raise FileNotFoundError(f"找不到 {p}，請依課前準備上傳課堂資料")

print("執行環境：", "Google Colab" if IN_COLAB else "本機")

In [ ]:
import urllib.request

import matplotlib.pyplot as plt
from matplotlib import font_manager

FONT_URL = "https://drive.google.com/uc?id=1eGAsTN1HBpJAkeVM57_C7ccp7hbgSz3_&export=download"
FONT_PATH = FONT_DIR / "TaipeiSansTCBeta-Regular.ttf"

if not FONT_PATH.exists():
    FONT_DIR.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(FONT_URL, FONT_PATH)

font_manager.fontManager.addfont(str(FONT_PATH))     # 告訴 Matplotlib：「這個字型檔可以使用」
plt.rcParams["font.family"] = "Taipei Sans TC Beta"  # 指定之後畫圖預設使用這個中文字型
plt.rcParams["axes.unicode_minus"] = False           # 避免負號 − 顯示異常
print("已設定中文字型：", FONT_PATH.name)

---

# Part A｜取得軌道車站與 YouBike 站點

## A1. 金鑰與 token：和 W6 相同

下面兩格就是 W6 的 C2、C3。`get_json` 每呼叫一次 API，就在 `呼叫紀錄` 記一筆，最後和 TDX 會員中心的使用統計比對。

In [ ]:
import os

if IN_COLAB:
    from google.colab import userdata
    for k in ["TDX_CLIENT_ID", "TDX_CLIENT_SECRET"]:
        try:
            os.environ[k] = userdata.get(k)
        except Exception as e:  # 沒設定密鑰，或沒打開 Notebook access
            print(f"讀不到 {k}：{type(e).__name__}")

HAS_KEY = all(os.environ.get(k) for k in ["TDX_CLIENT_ID", "TDX_CLIENT_SECRET"])
print("TDX 金鑰已載入" if HAS_KEY else "沒有 TDX 金鑰，Part A 將改讀老師的備援快照")

In [ ]:
import time

import requests

AUTH_URL = "https://tdx.transportdata.tw/auth/realms/TDXConnect/protocol/openid-connect/token"
API_BASE = "https://tdx.transportdata.tw/api/basic"

_token = {"value": None, "expires": 0}
呼叫紀錄 = []   # 每呼叫一次 API 記一筆，最後和 TDX 會員中心的使用統計比對


def get_token():
    """回傳 access token；還沒過期就沿用（token 快取）。"""
    if _token["value"] and time.time() < _token["expires"] - 60:
        return _token["value"]
    r = requests.post(AUTH_URL, data={
        "grant_type": "client_credentials",
        "client_id": os.environ["TDX_CLIENT_ID"],
        "client_secret": os.environ["TDX_CLIENT_SECRET"],
    }, timeout=30)
    r.raise_for_status()
    data = r.json()
    _token["value"] = data["access_token"]
    _token["expires"] = time.time() + data["expires_in"]
    return _token["value"]


def get_json(path, params=None):
    """呼叫 TDX 基礎服務，回傳解析後的 JSON。"""
    headers = {"authorization": f"Bearer {get_token()}", "Accept-Encoding": "gzip"}
    r = requests.get(API_BASE + path, headers=headers,
                     params={"$format": "JSON", **(params or {})}, timeout=30)
    呼叫紀錄.append({"path": path, "status": r.status_code})
    r.raise_for_status()
    return r.json()

## A2. 四份資料，一個函式

本週要四份資料。每一份都照 W6 的規則：**今天抓過就讀檔，不再呼叫 API**；沒有金鑰就讀老師的備援快照。把這個規則寫成函式，四份資料共用。

| 資料 | API | 說明 |
|---|---|---|
| 臺鐵車站 | `/v3/Rail/TRA/Station` | 全國一次取得，**沒有縣市欄位** |
| 高鐵車站 | `/v2/Rail/THSR/Station` | 全國一次取得 |
| 臺中捷運車站 | `/v2/Rail/Metro/Station/TMRT` | 其他縣市的捷運代碼見 A4 |
| 臺中市 YouBike 站點 | `/v2/Bike/Station/City/Taichung` | W6 用過的 API |

基礎會員每把金鑰**每分鐘最多 5 次**。下面這格第一次執行時呼叫 4 次，之後重跑都是讀檔。

In [ ]:
import json
from datetime import datetime
from zoneinfo import ZoneInfo

import geopandas as gpd
import pandas as pd

今天 = datetime.now(ZoneInfo("Asia/Taipei")).strftime("%Y%m%d")


def 讀取或下載(path, 名稱, 備援檔):
    """今天抓過就讀檔；有金鑰才呼叫 API 並存檔；都沒有就讀老師的備援快照。"""
    snap = TDX_DIR / f"{名稱}_{今天}.json"
    if snap.exists():
        print("讀取今天已存的快照：", snap.name)
        return json.loads(snap.read_text(encoding="utf-8"))
    if HAS_KEY:
        data = get_json(path)
        TDX_DIR.mkdir(parents=True, exist_ok=True)
        snap.write_text(json.dumps(data, ensure_ascii=False), encoding="utf-8")
        print("已從 TDX 取得並存檔：", snap.name)
        return data
    print("改讀老師的備援快照：", 備援檔.name)
    return json.loads(備援檔.read_text(encoding="utf-8"))


臺鐵 = 讀取或下載("/v3/Rail/TRA/Station", "rail_TRA", RAIL_BACKUP / "rail_TRA.json")
高鐵 = 讀取或下載("/v2/Rail/THSR/Station", "rail_THSR", RAIL_BACKUP / "rail_THSR.json")
捷運 = 讀取或下載("/v2/Rail/Metro/Station/TMRT", "metro_TMRT", RAIL_BACKUP / "metro_TMRT.json")
站點資料 = 讀取或下載("/v2/Bike/Station/City/Taichung", "station_Taichung", BIKE_BACKUP / "station_Taichung.json")

## A3. v3 的 JSON 外面多包了一層

W6 的站點 API 回傳 **list of dict**。臺鐵用的是 v3 版 API，先看它的型態，不要直接攤平：

In [ ]:
print("高鐵：", type(高鐵).__name__, len(高鐵), "筆")
print("臺鐵：", type(臺鐵).__name__, "，鍵：", list(臺鐵.keys()))

臺鐵回傳的是一個 **dict**：車站清單放在 `"Stations"` 裡，外層另外記了資料的更新時間。直接 `pd.json_normalize(臺鐵)` 只會得到 **1 列**：

In [ ]:
print(len(pd.json_normalize(臺鐵)), "列  ← 錯：把外層當成一筆資料")
print(len(臺鐵["Stations"]), "站 ← 對：車站清單在 Stations 裡")
print("臺鐵資料更新時間：", 臺鐵["UpdateTime"])

agent 如果照 v2 的格式寫程式，遇到 v3 會出現 `KeyError`，或者只得到 1 列卻沒有報錯。**先印出型態和鍵，再決定怎麼攤平。**

## A4. 其他縣市的捷運與輕軌（HW7 用）

以下代碼都可以接在 `/v2/Rail/Metro/Station/` 後面（老師 2026-10-23 實測）。臺鐵與高鐵都是全國一次取得，用 B2 的方法篩出你的縣市。

| 代碼 | 系統 | 車站所在縣市 |
|---|---|---|
| `TRTC` | 臺北捷運 | 臺北市、新北市 |
| `TRTCMG` | 貓空纜車 | 臺北市 |
| `NTMC` | 新北捷運（環狀線、三鶯線） | 新北市 |
| `NTDLRT` | 淡海輕軌 | 新北市 |
| `NTALRT` | 安坑輕軌 | 新北市 |
| `TYMC` | 桃園機場捷運 | 臺北市、新北市、桃園市 |
| `TMRT` | 臺中捷運 | 臺中市 |
| `KRTC` | 高雄捷運 | 高雄市 |
| `KLRT` | 高雄輕軌 | 高雄市 |

> 和 W6 一樣：**API 的系統不等於車站所在的縣市。** 桃園機場捷運 22 站中，只有 15 站在桃園市。

---

# Part B｜合併三種車站，篩出臺中市

## B1. 三份 JSON → 一個 GeoDataFrame

三份資料的欄位不完全相同（臺鐵沒有 `LocationCity`），所以只取需要的欄位：系統、代碼、站名、經緯度。

In [ ]:
def 轉成點(rows, 系統):
    """把 TDX 車站清單轉成 GeoDataFrame（EPSG:4326），只留系統、代碼、站名。"""
    df = pd.json_normalize(rows)
    return gpd.GeoDataFrame(
        {"系統": 系統, "StationUID": df["StationUID"], "站名": df["StationName.Zh_tw"]},
        geometry=gpd.points_from_xy(df["StationPosition.PositionLon"], df["StationPosition.PositionLat"]),
        crs="EPSG:4326",
    )


車站 = pd.concat([
    轉成點(臺鐵["Stations"], "臺鐵"),
    轉成點(高鐵, "高鐵"),
    轉成點(捷運, "臺中捷運"),
], ignore_index=True)

print(type(車站).__name__, 車站.crs, len(車站), "站")
車站["系統"].value_counts()

## B2. 臺鐵沒有縣市欄位：用 `sjoin` 判斷

臺鐵與高鐵都是全國的資料。要篩出臺中市，可以用 `StationAddress` 的地址字串，也可以用 W6 學的空間連接。地址的寫法不一定一致（「臺中」或「台中」、改制前的舊地名），**座標加上行政界線**比較可靠，還能順便得到行政區。

In [ ]:
town = gpd.read_file(TOWN_SHP)
車站 = 車站.to_crs(town.crs)   # EPSG:3826，單位公尺

車站 = gpd.sjoin(車站, town[["COUNTYNAME", "TOWNNAME", "geometry"]],
                 how="left", predicate="within").drop(columns="index_right")
print("沒有配對到鄉鎮的車站：", 車站["COUNTYNAME"].isna().sum())

臺中車站 = 車站[車站["COUNTYNAME"] == "臺中市"].reset_index(drop=True)
臺中車站.groupby("系統")["站名"].agg(["count", "、".join])

**【練習 B】** 改用地址篩選：臺鐵車站中，`StationAddress` 含有「臺中」的有幾站？和上面 `sjoin` 的結果一樣嗎？兩種方法結果一致，就是一次驗證。

In [ ]:
# TODO：從 臺鐵["Stations"] 篩出 StationAddress 含「臺中」的車站，和 sjoin 的結果比較

## B3. YouBike 站點 → GeoDataFrame

和 W6 的 D1、D2 相同，最後直接轉成 EPSG:3826。站點的站名欄改叫「站點名稱」，之後和車站的「站名」放在同一張表時才不會混淆。

In [ ]:
站點 = pd.json_normalize(站點資料).rename(columns={
    "StationName.Zh_tw": "站點名稱",
    "StationPosition.PositionLon": "經度",
    "StationPosition.PositionLat": "緯度",
})
站點 = gpd.GeoDataFrame(
    站點,
    geometry=gpd.points_from_xy(站點["經度"], 站點["緯度"]),
    crs="EPSG:4326",
).to_crs(town.crs)

print("站點數：", len(站點), "；CRS：", 站點.crs)
print("ServiceType：", 站點["ServiceType"].value_counts().to_dict(), "（2 = YouBike 2.0）")

In [ ]:
臺中 = town[town["COUNTYNAME"] == "臺中市"]

fig, ax = plt.subplots(figsize=(8, 7))
臺中.plot(ax=ax, color="whitesmoke", edgecolor="gray", linewidth=0.4)
站點.plot(ax=ax, color="lightgray", markersize=1, label="YouBike 站點")
for 系統, 顏色 in [("臺鐵", "tab:blue"), ("臺中捷運", "tab:green"), ("高鐵", "tab:orange")]:
    臺中車站[臺中車站["系統"] == 系統].plot(ax=ax, color=顏色, markersize=25, label=系統)
ax.set_xlim(195000, 235000)   # 只看平地，和平區的山區先略過
ax.set_ylim(2650000, 2705000)
ax.set_title("臺中市的軌道車站與 YouBike 站點")
ax.legend(loc="lower right")
ax.set_axis_off()
plt.show()

---

# Part C｜環域（buffer）與單位

## C1. 在經緯度下 `buffer(500)`

`buffer(距離)` 會在每個點周圍畫一個圓，**距離的單位就是 CRS 的單位**。先故意在經緯度（EPSG:4326）下做一次：

In [ ]:
import warnings

臺中車站_經緯度 = 臺中車站.to_crs("EPSG:4326")

with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always")
    錯的環域 = 臺中車站_經緯度.buffer(500)

for x in w:
    print("警告：", str(x.message)[:100])
print("環域範圍（經度、緯度）：", 錯的環域.total_bounds.round(1))

經度範圍變成大約 −380 到 620、緯度 −476 到 525：這是半徑 500「度」的圓，可是地球的經度只有 −180 到 180。**程式沒有報錯，只有一個警告。** 拿這個環域去計數，每個車站都會「包含」臺中市全部的站點：

In [ ]:
錯的計數 = gpd.sjoin(站點.to_crs("EPSG:4326")[["geometry"]],
                     gpd.GeoDataFrame(geometry=錯的環域), predicate="within")
print("每個車站「500 單位內」的站點數：", 錯的計數.groupby("index_right").size().unique())

## C2. 先轉成 EPSG:3826，再 buffer

`臺中車站` 在 B2 已經是 EPSG:3826（單位公尺）。用**面積反推半徑**，是確認單位最快的方法：

In [ ]:
import numpy as np

print("CRS 的單位：", 臺中車站.crs.axis_info[0].unit_name)

環域500 = 臺中車站[["系統", "站名", "geometry"]].copy()
環域500["geometry"] = 臺中車站.buffer(500)

print("一個環域的面積：", round(環域500.area.iloc[0]), "平方公尺")
print("π × 500²      ：", round(np.pi * 500**2), "平方公尺")

環域其實是用多邊形逼近圓（預設每 1/4 圓用 16 段），所以面積比 π × 500² 小一點點，差不到 0.2%。

## C3. 改用 0.005 度可以嗎？

有人會想：「500 公尺大約是 0.005 度，在經緯度下 `buffer(0.005)` 就好。」畫出來量量看：

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore")   # 經緯度下 buffer 的警告，C1 已經看過
    近似 = 臺中車站_經緯度.buffer(0.005).to_crs("EPSG:3826")   # 先在經緯度下畫，再轉回公尺量大小

x0, y0, x1, y1 = 近似.iloc[0].bounds
print(f"東西寬 {x1 - x0:.0f} 公尺、南北高 {y1 - y0:.0f} 公尺")

它不是圓，是**橢圓**：經線往北會收攏，所以經度 1 度的長度隨緯度變短，緯度 1 度則大致固定。同樣的 0.005 度，到了北部、南部又是不同的公尺數。

**【練習 C】** 用下面的近似值，算出 0.005 度在臺中（北緯約 24.15 度）的東西、南北各是幾公尺，和上面量到的寬、高（÷ 2）比較。

- 緯度 1 度 ≈ 110.7 公里
- 經度 1 度 ≈ 111.32 × cos(緯度) 公里

In [ ]:
# TODO：用 np.cos(np.radians(緯度)) 計算
緯度 = 24.15

---

# Part D｜環域內有幾個 YouBike 站

## D1. 站點 `sjoin` 環域

左邊放站點、右邊放環域，`predicate="within"`。`sjoin` 預設是 inner join，**只留下落在某個環域內的站點**，所以連接後的筆數比站點少很多。

反過來，一個站點如果同時落在兩個車站的環域裡（例如大慶站的臺鐵與捷運），會出現兩列。這次是正常的：我們要的就是「每個車站各有幾個」。

In [ ]:
連接 = gpd.sjoin(站點[["StationUID", "站點名稱", "BikesCapacity", "geometry"]], 環域500, predicate="within")
print("站點數：", len(站點), "→ 連接後：", len(連接))
連接.head()

`index_right` 是環域（也就是車站）的列號，用它分組計數，再對應回每個車站。

**W6 的陷阱又來了：** 環域內沒有任何站點的車站，不會出現在 `groupby` 的結果裡。用 `reindex` 補回來，記為 0。把這段寫成函式，換距離、換縣市都能重用：

In [ ]:
def 環域計數(車站, 站點, 公尺):
    """回傳每個車站周圍「公尺」範圍內的站點數；沒有站點的車站記為 0。"""
    環域 = 車站[["geometry"]].copy()
    環域["geometry"] = 車站.buffer(公尺)
    連接 = gpd.sjoin(站點[["geometry"]], 環域, predicate="within")
    return 連接.groupby("index_right").size().reindex(車站.index, fill_value=0)


臺中車站["300m站點數"] = 環域計數(臺中車站, 站點, 300)
臺中車站["500m站點數"] = 環域計數(臺中車站, 站點, 500)

print("500 公尺內沒有站點的車站：", (臺中車站["500m站點數"] == 0).sum(), "站")
print("300 公尺內沒有站點的車站：", (臺中車站["300m站點數"] == 0).sum(), "站")
臺中車站.sort_values("300m站點數")[["系統", "站名", "TOWNNAME", "300m站點數", "500m站點數"]].head(8)

**臺中市 42 個車站，每一站 300 公尺內都有 YouBike。** 這個結果在 Part F 討論。

**【練習 D】** 用 `環域計數` 算 100 公尺與 200 公尺，各有幾個車站周圍沒有站點？

In [ ]:
# TODO：用 for 迴圈分別計算 100、200 公尺

---

# Part E｜最近鄰連接 `sjoin_nearest`

## E1. 每個車站到最近站點的距離

環域只能回答「範圍內有沒有、有幾個」。`sjoin_nearest` 直接回答「**最近的在多遠**」：`distance_col` 會多一個距離欄位，單位同樣是 CRS 的單位。

In [ ]:
最近 = gpd.sjoin_nearest(
    臺中車站[["系統", "站名", "geometry"]],
    站點[["站點名稱", "geometry"]],
    how="left",
    distance_col="最近距離_m",
)
print("車站數：", len(臺中車站), "→ 連接後：", len(最近))
最近.sort_values("最近距離_m", ascending=False).head(5)

## E2. 筆數會變多嗎？

如果有兩個站點和車站**一樣近**，`sjoin_nearest` 會兩個都回傳，那個車站就變成兩列。臺中這次沒有發生，你的縣市不一定。每次都先檢查，再用 `groupby(level=0)` 安全地對應回車站：

In [ ]:
print("重複出現的車站：", 最近.index.duplicated().sum())

臺中車站["最近距離_m"] = 最近.groupby(level=0)["最近距離_m"].min().round(0)
臺中車站["最近站點"] = 最近.groupby(level=0)["站點名稱"].first()
臺中車站["最近距離_m"].describe().round(0)

## E3. `max_distance`：另一種找「500 公尺內沒有」的方法

加上 `max_distance=500`，超過 500 公尺就不配對，`how="left"` 會留下空值。和 D 的結果一致，就是一次交叉驗證。

In [ ]:
五百內 = gpd.sjoin_nearest(臺中車站[["站名", "geometry"]], 站點[["站點名稱", "geometry"]],
                           how="left", max_distance=500)
print("500 公尺內找不到站點的車站：", 五百內["站點名稱"].isna().sum(), "站")

**【練習 E】** 把兩個圖層都轉成 EPSG:4326，再做一次 `sjoin_nearest`。距離欄的數字是多少？單位是什麼？有沒有警告？

In [ ]:
# TODO：to_crs("EPSG:4326") 之後再 sjoin_nearest，觀察距離欄與警告

---

# Part F｜臺中的答案是 0：換指標、換縣市

## F1. 門檻沒有鑑別力

In [ ]:
print("500 公尺內沒有站點的車站：", (臺中車站["500m站點數"] == 0).sum(), "站")
print("最近距離：中位數", 臺中車站["最近距離_m"].median(), "公尺；最遠", 臺中車站["最近距離_m"].max(), "公尺")
臺中車站.nlargest(3, "最近距離_m")[["系統", "站名", "最近站點", "最近距離_m"]]

臺中市的 YouBike 站點很密，**每個車站 300 公尺內都有站點**。「哪些車站 500 公尺內沒有 YouBike？」在臺中的答案是 0，這個指標分不出好壞。這時有兩條路：

1. **換指標：** 從「有沒有」改問「夠不夠」，例如 300 公尺內的**車柱數**（`BikesCapacity`），代表可以停放與借出的容量。
2. **換地方：** 同樣的程式，在站點稀疏的縣市，答案完全不同（F3）。

## F2. 300 公尺內的車柱數

In [ ]:
環域300 = 臺中車站[["geometry"]].copy()
環域300["geometry"] = 臺中車站.buffer(300)
連接300 = gpd.sjoin(站點[["BikesCapacity", "geometry"]], 環域300, predicate="within")

臺中車站["300m車柱數"] = (連接300.groupby("index_right")["BikesCapacity"].sum()
                         .reindex(臺中車站.index, fill_value=0))
臺中車站.sort_values("300m車柱數")[["系統", "站名", "300m站點數", "300m車柱數", "最近距離_m"]].head(8)

車柱數最少的車站，是「有站點但容量很小」的地方。注意：**彰化縣、雲林縣的 Moovo 沒有 `BikesCapacity`**，這兩個縣市只能用站點數。

## F3. 換到屏東縣

臺鐵、高鐵資料是全國的，已經在 `車站` 裡；屏東縣的站點直接讀 W6 的備援快照，**不必再呼叫 API**。`環域計數` 函式原封不動地重用：

In [ ]:
屏東資料 = json.loads((BIKE_BACKUP / "station_PingtungCounty.json").read_text(encoding="utf-8"))
屏東站點 = pd.json_normalize(屏東資料)
屏東站點 = gpd.GeoDataFrame(
    屏東站點,
    geometry=gpd.points_from_xy(屏東站點["StationPosition.PositionLon"], 屏東站點["StationPosition.PositionLat"]),
    crs="EPSG:4326",
).to_crs(town.crs)

屏東車站 = 車站[車站["COUNTYNAME"] == "屏東縣"].reset_index(drop=True)
屏東車站["500m站點數"] = 環域計數(屏東車站, 屏東站點, 500)

print("屏東縣：車站", len(屏東車站), "站、YouBike 站點", len(屏東站點), "站")
print("500 公尺內沒有站點的車站：", (屏東車站["500m站點數"] == 0).sum(), "站")
屏東車站.loc[屏東車站["500m站點數"] == 0, ["系統", "站名", "TOWNNAME"]]

其中一站叫「**潮州基地**」：它是臺鐵的車輛基地，不是給旅客上下車的車站。**API 給的清單，不一定等於你要分析的對象**（Part H）。

**【練習 F】** 把 F3 的屏東縣換成你 HW6 選的縣市（`BIKE_BACKUP` 裡有 15 個縣市的站點快照），500 公尺內沒有站點的車站有幾站？如果你的縣市有捷運，記得用 A4 的代碼另外取得。

In [ ]:
# TODO：仿照 F3，把縣市名稱與代碼換成你 HW6 選的縣市
我的縣市, 我的代碼 = "苗栗縣", "MiaoliCounty"   # 換成你的縣市

---

# Part G｜`dissolve` 與 `overlay`：各區有多少面積在車站 500 公尺內？

## G1. 環域重疊，面積會被重複計算

In [ ]:
各自相加 = 環域500.area.sum() / 1e6
聯集 = 環域500.dissolve()   # 全部合併成一個（Multi）Polygon，重疊的地方只算一次
聯集面積 = 聯集.area.iloc[0] / 1e6

print(f"{len(環域500)} 個環域的面積直接相加：{各自相加:.2f} 平方公里")
print(f"dissolve 之後：                {聯集面積:.2f} 平方公里")
print(f"重複計算的面積：               {各自相加 - 聯集面積:.2f} 平方公里")

大慶站的臺鐵與捷運只差 36 公尺，兩個環域幾乎完全重疊；市區車站間距不到 1 公里的地方也會重疊。要算「被覆蓋的面積」，一定要先 `dissolve`。

## G2. `overlay`：環域聯集 × 行政區

`overlay(..., how="intersection")` 會把兩個面圖層切開，留下重疊的部分，並帶著雙方的屬性。兩個圖層的 CRS 必須相同。

In [ ]:
臺中區 = town[town["COUNTYNAME"] == "臺中市"][["TOWNCODE", "TOWNNAME", "geometry"]].copy()
臺中區["面積_km2"] = 臺中區.area / 1e6

交集 = gpd.overlay(臺中區, 聯集[["geometry"]], how="intersection")
交集["覆蓋_km2"] = 交集.area / 1e6
print("臺中市", len(臺中區), "區；和環域有交集的：", len(交集), "區")

臺中區["覆蓋_km2"] = 臺中區["TOWNCODE"].map(交集.set_index("TOWNCODE")["覆蓋_km2"]).fillna(0)
臺中區["覆蓋比例"] = 臺中區["覆蓋_km2"] / 臺中區["面積_km2"]
臺中區.sort_values("覆蓋比例", ascending=False).head(8).round(3)

沒有車站的區不會出現在 `交集` 裡，所以用 `map` 對應回去後要 `fillna(0)`：**0 的陷阱第三次出現**（W6 的鄉鎮計數、D1 的環域計數、這裡）。

**【練習 G】** 驗證：所有區的 `覆蓋_km2` 加起來，和 G1 的聯集面積一樣嗎？在你的縣市，什麼情況下兩者會不相等？

In [ ]:
# TODO：比較 臺中區["覆蓋_km2"].sum() 與 聯集面積

## G3. 地圖

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
臺中區.plot(column="覆蓋比例", cmap="Blues", legend=True, ax=ax, edgecolor="gray", linewidth=0.4,
            legend_kwds={"label": "車站 500 公尺範圍占全區面積比例", "shrink": 0.6})
聯集.boundary.plot(ax=ax, color="tab:red", linewidth=0.8)
臺中車站.plot(ax=ax, color="black", markersize=6)
ax.set_xlim(195000, 235000)
ax.set_ylim(2650000, 2705000)
ax.set_title("臺中市各區的軌道車站 500 公尺覆蓋比例")
ax.set_axis_off()
plt.show()

In [ ]:
%pip install -q mapclassify

In [ ]:
# 臺灣通用電子地圖（內政部國土測繪中心 WMTS）；注意網址是 {z}/{y}/{x}
底圖網址 = "https://wmts.nlsc.gov.tw/wmts/EMAP/default/GoogleMapsCompatible/{z}/{y}/{x}"
底圖出處 = "臺灣通用電子地圖 © 內政部國土測繪中心"

m = 環域500.explore(color="tab:red", style_kwds={"fillOpacity": 0.1, "weight": 1},
                    tooltip=["系統", "站名"], tiles=底圖網址, attr=底圖出處)
站點.explore(m=m, color="gray", marker_kwds={"radius": 2}, tooltip=["站點名稱", "BikesCapacity"])
臺中車站.explore(m=m, column="系統", marker_kwds={"radius": 6},
                 tooltip=["系統", "站名", "300m站點數", "300m車柱數", "最近站點", "最近距離_m"])
m

---

# Part H｜什麼算「一個車站」？直線距離的限制

## H1. 共站：一個轉乘站，資料裡有好幾個點

用 `sjoin_nearest` 找每個車站最近的**另一個車站**（`exclusive=True`：不要配到自己）：

In [ ]:
鄰近 = gpd.sjoin_nearest(
    臺中車站[["系統", "站名", "geometry"]],
    臺中車站[["系統", "站名", "geometry"]],
    distance_col="距離_m",
    exclusive=True,
)
鄰近 = 鄰近[鄰近["距離_m"] < 400].sort_values("距離_m")
鄰近[["系統_left", "站名_left", "系統_right", "站名_right", "距離_m"]].round(0)

每一對都會出現兩次（A 找到 B、B 也找到 A）。

- **大慶、松竹：** 臺鐵與捷運各一個點，相距不到 100 公尺。
- **新烏日、高鐵臺中站：** 臺鐵與捷運相距 84 公尺。
- **高鐵「台中」與捷運「高鐵臺中站」：** 相距 327 公尺，名字一個寫「台」、一個寫「臺」，**用站名比對根本合併不到**。

計算「有幾個車站缺 YouBike」或「平均距離」時，要先決定共站算一站還是多站，並寫進決策表。只算覆蓋面積的話，`dissolve` 已經處理了重疊。

## H2. 清單裡不是車站的東西

臺鐵清單有 245 筆。名稱裡有「基地」或「-」的：

In [ ]:
臺鐵表 = pd.json_normalize(臺鐵["Stations"])
臺鐵表.loc[臺鐵表["StationName.Zh_tw"].str.contains("基地|-"),
          ["StationID", "StationName.Zh_tw", "StationClass", "StationAddress"]]

- **臺北-環島：** 和臺北站的座標只差 1 公尺，是環島列車使用的代碼，不是另一個車站。
- **潮州基地：** 臺鐵的車輛基地。

臺北捷運的資料也有類似的情況：**轉乘站依路線各列一次**。讀備援快照看看（不必呼叫 API）：

In [ ]:
北捷 = 轉成點(json.loads((RAIL_BACKUP / "metro_TRTC.json").read_text(encoding="utf-8")), "臺北捷運")
print("臺北捷運：", len(北捷), "筆；站名不重複的有", 北捷["站名"].nunique(), "個")
北捷[北捷["站名"].duplicated(keep=False)].sort_values("站名").head(6)

台北車站有 BL12、R10 兩筆，**座標完全相同**。不先處理的話，一個 YouBike 站點會被同一個轉乘站算兩次。

注意：H1 的 `exclusive=True` 會跳過「和自己**完全相同**的幾何」，所以座標一模一樣的重複點，用 H1 的方法**找不到**。要另外用 `duplicated()` 檢查：

In [ ]:
print("座標完全相同的重複點：", 北捷.geometry.duplicated().sum(), "個")
print("臺中市的車站：", 臺中車站.geometry.duplicated().sum(), "個")

## H3. 直線距離的限制

環域與 `sjoin_nearest` 都是**直線距離**。實際走路要沿著道路，遇到鐵路、河川、快速道路或圍牆就得繞路，直線 200 公尺可能要走 600 公尺。本課程只做直線距離，但**結論中要說明這個限制**。

**【練習 H】** 在 G3 的互動地圖上，放大「最近距離」最遠的 3 個車站（F1 的表），看看最近的站點和車站之間有沒有阻隔，例如要從車站的另一側出口出去、或要過一條大馬路。把觀察寫進你的筆記。

---

# 存檔與 API 用量

In [ ]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
GPKG = PROCESSED_DIR / "w7_taichung_access.gpkg"

臺中車站.to_file(GPKG, layer="stations", driver="GPKG")
聯集.to_file(GPKG, layer="buffer500_union", driver="GPKG")
臺中區.to_file(GPKG, layer="town_coverage", driver="GPKG")

import pyogrio
print(pyogrio.list_layers(GPKG))

In [ ]:
print("本次執行呼叫 API：", len(呼叫紀錄), "次（不含換 token）")
pd.DataFrame(呼叫紀錄)